# Creating Size-Dependent States and Operators

The [State](../apidoc/_autosummary/pulser.backend.State.rst) and [Operator](../apidoc/_autosummary/pulser.backend.Operator.rst) classes let you build arbitrary state and operator objects from scratch using the `from_state_amplitudes()` and `from_operator_repr()` methods (as shown in [Execution on an Emulator](backends.nblink) and [Results and Observables](../results.ipynb)). These objects are used to create initial states, calculate observables such as `Fidelity`, and find expectation values of custom operators at the end of your Pulser simulation.

A common use case for these is many-body physics simulations, where you often want to:

- prepare reference initial states that are non-trivial,
- scale the number of qubits `n_qubits` to look for trends that persist (or not).

This page presents some simple recipes for defining commonly used states and operators with a variable system size.

## Size-dependent helper function for Pauli operators

For a two-level system with `eigenstates = ("r", "g")` (corresponding to the [ground-rydberg basis](../conventions.md#pauli-matrix-form)), the Pauli matrices are written as:

In [ ]:
from collections.abc import Collection
from typing import Type
import numpy as np
import matplotlib.pyplot as plt

from pulser.backend import Operator, OperatorRepr
from pulser_simulation import QutipConfig

eigenstates = ("r", "g")

sigma_repr = {
    "x": {"gr": 1.0, "rg": 1.0},
    "y": {"gr": 1.0j, "rg": -1.0j},
    "z": {"rr": 1.0, "gg": -1.0},
}

Instead of repeating `eigenstates` and `n_qudits` at every call, it is convenient to write small helper functions parametrized by the number of qubits (`n_qubits`). The `multi_pauli_op()` function below builds the operator for a product of identical Pauli matrices acting on an arbitrary subset of sites, e.g. `multi_pauli_op(n_qubits=4, pauli="x", idxs=[0, 2])` builds $X_0 X_2$.

In [ ]:
def multi_pauli_op(
    n_qubits: int,
    operator_class: Type[Operator] = OperatorRepr,
    pauli: str = "x",
    idxs: int | Collection[int] = 0,
) -> Operator:
    """Builds the operator for a product of identical Pauli matrices.

    Mixed Pauli strings, such as X_0 Z_1, are not supported.

    Args:
        n_qubits: The number of qubits in the system.
        operator_class: The ``Operator`` subclass to instantiate, usually
            taken from a backend's ``config_type.operator_type``.
        pauli: The Pauli matrix to apply, one of "x", "y" or "z".
        idxs: The index (or indices) of the qubits the Pauli matrix acts
            on. Duplicate indices are ignored and the remaining qubits
            are applied the identity.

    Returns:
        The operator for the requested Pauli string.

    Raises:
        ValueError: If ``pauli`` is not a known Pauli matrix.

    Examples:
        >>> # builds X_0 X_2 on a system of four qubits
        >>> multi_pauli_op(n_qubits=4, pauli="x", idxs=[0, 2])
    """
    if pauli not in sigma_repr:
        raise ValueError(
            f"'pauli' must be one of {tuple(sigma_repr)}; got {pauli!r}."
        )

    # Both a single int and a collection of ints are accepted as 'idxs';
    # the conversion to a set also discards any duplicate index.
    idx = {idxs} if isinstance(idxs, int) else set(idxs)

    # A single tensor operator with coefficient 1.0, applying the same
    # single-qubit operator to every index in 'idx'.
    return operator_class.from_operator_repr(
        eigenstates=eigenstates,
        n_qudits=n_qubits,
        operations=[(1.0, [(sigma_repr[pauli], idx)])],
    )

The operator class to instantiate is taken from the backend's configuration class (`QutipConfig` is used below as an example), so that the same helper works with any backend. Leaving `operator_class` at its `OperatorRepr` default, or picking any other `Operator` subclass, is equally valid. Since the operator is built through `from_operator_repr()`, it stays compatible with remote backends either way.

In [ ]:
operator_class = QutipConfig.operator_type

# X_0, with a single index given as an int
x0 = multi_pauli_op(
    n_qubits=4, operator_class=operator_class, pauli="x", idxs=0
)
# Z_1 Z_3, with the indices given as a list
z1z3 = multi_pauli_op(
    n_qubits=4, operator_class=operator_class, pauli="z", idxs=[1, 3]
)

z1z3.to_qobj()

## Building reference states

In the same spirit, `get_state()` builds reference states of arbitrary size via `from_state_amplitudes()`, which takes a mapping between basis state combinations and their complex amplitudes.

In [ ]:
from pulser.backend import State, StateRepr

def get_state(
    n_qubits: int,
    state_class: Type[State] = StateRepr,
    state_name: str = "plus",
) -> State:
    """Builds a reference state of size n_qubits.

    Args:
        n_qubits: The number of qubits in the state.
        state_class: The ``State`` subclass to instantiate, usually taken
            from a backend's ``config_type.state_type``.
        state_name: The reference state to build, either "plus" (the
            uniform superposition of all basis states) or "ghz".

    Returns:
        The requested reference state, normalized to 1.

    Raises:
        NotImplementedError: If ``state_name`` is not a known reference
            state.

    Examples:
        >>> # builds (|rrrr> + |gggg>) / sqrt(2)
        >>> get_state(n_qubits=4, state_name="ghz")
    """
    if state_name == "plus":
        # |+>^n_qubits: all 2^n_qubits basis states with equal amplitude
        dim = 2**n_qubits
        amplitudes = {
            # format(i, "0{n}b") writes the integer i as a binary string
            # of fixed length n_qubits, whose digits then index into
            # 'eigenstates' to name the basis state combination
            "".join(
                eigenstates[int(b)] for b in format(i, f"0{n_qubits}b")
            ): 1.0
            / np.sqrt(dim)
            for i in range(dim)
        }

    elif state_name == "ghz":
        # (|rr...r> + |gg...g>) / sqrt(2): only two basis states have a
        # non-zero amplitude, so they are the only ones to list
        norm = 1.0 / np.sqrt(2)
        amplitudes = {"r" * n_qubits: norm, "g" * n_qubits: norm}

    else:
        raise NotImplementedError(f"State {state_name} not implemented.")

    return state_class.from_state_amplitudes(
        eigenstates=eigenstates, amplitudes=amplitudes
    )

Both helpers depend on the system size only through their `n_qubits` argument, so the exact same code can be reused for any number of qubits. Below, the `"plus"` and `"ghz"` states are built for the same `n_qubits` and compared through their bitstring probabilities, obtained with `bitstring_probabilities()`.

In [ ]:
state_class = QutipConfig.state_type

n_qubits = 4
plus_state = get_state(
    n_qubits=n_qubits, state_class=state_class, state_name="plus"
)
ghz_state = get_state(
    n_qubits=n_qubits, state_class=state_class, state_name="ghz"
)

# bitstring_probabilities() omits the bitstrings with zero probability,
# so the full list is built here to share the x-axis between both plots
bitstrings = [format(i, f"0{n_qubits}b") for i in range(2**n_qubits)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
titles = (r"$|+\rangle^{\otimes n_\mathrm{qubits}}$", "GHZ")

for ax, state, title in zip(axes, (plus_state, ghz_state), titles):
    probs = state.bitstring_probabilities()
    ax.bar(bitstrings, [probs.get(b, 0.0) for b in bitstrings])
    ax.set_title(title)
    ax.set_xlabel("bitstring")
    ax.tick_params(axis="x", rotation=90)

axes[0].set_ylabel("probability")
fig.tight_layout()